# $100 -> $1,000: a trading agent, backtested honestly

This notebook starts a trading account with **$100** and asks whether an automated agent can
compound it to **$1,000** - a 10x - and how much risk that takes.

**Read this part before the numbers.** Turning $100 into $1,000 means a +900% return. Nothing
makes that easy, and the two ways a backtest can fake it are both handled explicitly here:

1. **Lookahead.** Using information the agent could not have had. The whole package is built
   so a decision made at bar `t` is filled at the **open of bar `t+1`**, and the test-suite
   proves every indicator, strategy and blend is causal by truncating the future and checking
   nothing in the past moves.
2. **Overfitting.** Trying hundreds of settings and reporting the winner. Here parameters are
   chosen on a **training window** and then traded untouched on the **following window**
   (walk-forward). Every number reported as a result comes from data the optimiser had not seen.

Costs are charged the way an exchange charges them: a fee and slippage on every fill, financing
on leverage, and financing on shorts. On a $100 account those costs are not a rounding error -
they are one of the main things standing between you and the target.

**This is research code for backtesting. It is not financial advice, and it does not place orders.**
Past performance - especially simulated past performance - does not predict future returns. Crypto
can and does fall 80%. Never trade money you cannot afford to lose entirely.

## 0. Setup

Clones the repo (on Colab) or finds it locally, then installs dependencies.

In [ ]:
import os, subprocess, sys

REPO = 'https://github.com/Blobby132/Trading-agent.git'
BRANCH = 'claude/trading-agent-backtest-sih2te'

def find_root():
    here = os.getcwd()
    for path in (here, os.path.dirname(here), os.path.join(here, 'Trading-agent')):
        if os.path.isdir(os.path.join(path, 'tradingagent')):
            return os.path.abspath(path)
    return None

root = find_root()
if root is None:
    subprocess.run(['git', 'clone', '-q', '--branch', BRANCH, REPO, 'Trading-agent'], check=True)
    root = os.path.abspath('Trading-agent')
os.chdir(root)
sys.path.insert(0, root)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=False)
print('working in', root)

In [ ]:
import warnings
from dataclasses import replace

warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tradingagent.data import load_prices, load_universe, align_universe, bars_per_year
from tradingagent.agent import AgentConfig, TradingAgent, PortfolioAgent
from tradingagent.engine import ExecutionConfig, buy_and_hold_equity
from tradingagent.risk import RiskConfig
from tradingagent.optimize import walk_forward, WalkForwardConfig, DEFAULT_SEARCH_SPACE
from tradingagent.metrics import format_summary, monte_carlo_paths, deflated_sharpe, time_to_target
from tradingagent.report import tearsheet, plot_equity, plot_folds, fold_table, param_frequency
from tradingagent.live import recommend

pd.set_option('display.width', 140)

START_CAPITAL = 100.0    # what we begin with
TARGET        = 1000.0   # what we are trying to reach
SYMBOL        = 'BTC-USD'
INTERVAL      = '1d'
START_DATE    = '2015-01-01'
PPY           = bars_per_year(INTERVAL)
print('ready')

## 1. Data

Daily candles from the **Coinbase Exchange public API** - no API key, no account, and the same
venue whose fees we are charging ourselves. `load_prices` caches to `data/cache/`, so re-running
a cell does not re-download.

Crypto is the honest choice for a $100 account: it trades 24/7, it is divisible to eight decimal
places (so $100 can actually hold a position), and it is volatile enough that a 10x is not absurd
on the timescale of a few years. That same volatility is what can also take the account to zero.

In [ ]:
prices = load_prices(SYMBOL, interval=INTERVAL, start=START_DATE, source='coinbase')
print(f'{len(prices):,} bars   {prices.index[0].date()} -> {prices.index[-1].date()}')
display(prices.tail(3).round(2))

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(prices.index, prices['close'], color='#2a78d6', linewidth=1.4)
ax.set_yscale('log'); ax.set_title(f'{SYMBOL} close (log scale)', loc='left')
ax.grid(color='#e3e2de'); ax.set_axisbelow(True)
for s in ('top','right'): ax.spines[s].set_visible(False)
plt.show()

## 2. The number that does not count

First, the tempting version: run the default agent over the **entire history at once**. This is
what most backtests report, and it is close to meaningless - the settings were picked by someone
(me) who had already seen this data.

It is here as a reference point and as a reminder of what to distrust.

In [ ]:
exec_cfg = ExecutionConfig(
    initial_capital=START_CAPITAL,
    target_equity=TARGET,
    fee_bps=10.0,        # 0.10% per side
    slippage_bps=5.0,    # 0.05% per side
    max_leverage=2.0,
    periods_per_year=PPY,
)
risk_cfg = RiskConfig(target_vol=0.50, max_leverage=2.0)

in_sample = TradingAgent(AgentConfig(periods_per_year=PPY), risk_cfg).backtest(prices, exec_cfg)
benchmark = buy_and_hold_equity(prices, START_CAPITAL, exec_cfg.fee_bps)
print(format_summary(in_sample.stats(benchmark=benchmark), 'IN-SAMPLE - do not trust this'))

## 3. The number that does count: walk-forward

The agent is re-fitted as it goes, exactly as it would have to be in real life:

```
|<--- train 730 bars --->|<-embargo->|<-- trade 182 bars -->|
                                   |<--- train 730 bars --->|<-embargo->|<-- trade 182 -->|
```

On each training window the optimiser tries `n_candidates` random configurations - which
strategies to run, how to blend them, how much volatility to target, how much leverage, how wide
the stops, how much drawdown to tolerate - and scores them on a **growth objective that penalises
deep drawdowns and disqualifies anything that drew down more than 45% in training**. The best
`top_k` configurations are then *averaged* and traded, untouched, over the next six months.
Equity carries across windows, so what comes out is a single continuously-compounding account.

The search never sees the window it trades. Every result below is out of sample.

In [ ]:
wf_cfg = WalkForwardConfig(
    train_bars=730,      # ~2 years to fit on
    test_bars=182,       # ~6 months traded out of sample
    embargo_bars=10,     # gap so rolling windows cannot leak across the join
    n_candidates=150,    # configurations searched per fold
    top_k=5,             # best configurations blended into what actually trades
    objective='target_growth',
    max_train_drawdown=0.45,
    seed=1,
)

wf = walk_forward(prices, exec_cfg, wf_cfg)

In [ ]:
bench_oos = benchmark.reindex(wf.equity.index)
stats = wf.stats(benchmark=bench_oos)
print(format_summary(stats, f'{SYMBOL} WALK-FORWARD (out of sample)'))

In [ ]:
fig = tearsheet(
    wf.equity,
    benchmark=bench_oos,
    weights=wf.weights,
    folds=wf.folds,
    returns=wf.returns,
    target=TARGET,
    initial=START_CAPITAL,
    title=f'{SYMBOL} - walk-forward, ${START_CAPITAL:,.0f} start',
)
plt.show()

## 4. Did the account reach $1,000?

The brief was not "beat an index" - it was **reach $1,000**. That is a question about a path,
not about an average return, so it gets answered directly: when did equity first close at or
above the target, and what did the account have to live through on the way?

In [ ]:
tt = time_to_target(wf.equity, TARGET, PPY)
if tt['target_hit']:
    print(f"Target ${TARGET:,.0f} first reached on {tt['date_to_target'].date()} - "
          f"{int(tt['bars_to_target']):,} bars ({tt['years_to_target']:.2f} years) after the first trade.")
    print(f"Worst drawdown suffered along the way: {stats['max_drawdown']:.1%}")
    print(f"Deepest the account ever got before reaching it: ${wf.equity.loc[:tt['date_to_target']].min():,.2f}")
else:
    peak = wf.equity.max()
    print(f"Target NOT reached. Best the account ever did: ${peak:,.2f} "
          f"({peak / START_CAPITAL:.1f}x) on {wf.equity.idxmax().date()}.")
print()
display(fold_table(wf.folds))

### Which settings did the optimiser keep choosing?

A parameter whose winning value is different every fold is noise being chased. One that gets
picked over and over is closer to a real property of the market. This is a cheap and surprisingly
revealing overfitting check.

In [ ]:
freq = param_frequency(wf.chosen)
top = freq.sort_values(['param','times_chosen'], ascending=[True, False]).groupby('param').head(2)
display(top.reset_index(drop=True))

## 5. Is the edge real, or did we get lucky?

Three checks, in increasing order of how uncomfortable they are.

**a) Seed sensitivity.** The search is random. If the result only appears for one random seed,
there is no result.

In [ ]:
seed_rows = []
for seed in [1, 2, 3, 4, 5]:
    r = walk_forward(prices, exec_cfg, replace(wf_cfg, seed=seed, verbose=False))
    s = r.stats()
    seed_rows.append({
        'seed': seed,
        'final equity': s['final_equity'],
        'sharpe': s['sharpe'],
        'max drawdown': s['max_drawdown'],
        'reached target': bool(s.get('target_hit', 0)),
        'target date': str(s.get('date_to_target', ''))[:10],
    })
seeds_df = pd.DataFrame(seed_rows)
display(seeds_df.style.format({'final equity': '${:,.0f}', 'sharpe': '{:.2f}', 'max drawdown': '{:.1%}'}))
print(f"median final equity ${seeds_df['final equity'].median():,.0f}   "
      f"reached target in {seeds_df['reached target'].mean():.0%} of seeds")

**b) Bootstrapped alternative histories.** The realised out-of-sample returns are resampled in
blocks (which preserves streaks and volatility clustering) into thousands of other futures that
share the same edge. The spread is the honest uncertainty around the headline number - and it
prices the outcome nobody puts in a pitch deck: how often the account goes to zero.

In [ ]:
mc = monte_carlo_paths(wf.returns, initial_capital=START_CAPITAL, target=TARGET, n_paths=5000, seed=7)
print(f"across {int(mc['n_paths']):,} bootstrapped histories of the same length:")
print(f"  reached ${TARGET:,.0f}       {mc['p_hit_target']:6.1%}")
print(f"  wiped out             {mc['p_ruin']:6.1%}")
print(f"  final equity   5th ${mc['p05_final_equity']:>9,.0f}   median ${mc['median_final_equity']:>9,.0f}   95th ${mc['p95_final_equity']:>9,.0f}")
print(f"  typical worst drawdown  {mc['median_max_drawdown']:.1%}")

**c) Deflated Sharpe.** Search 150 configurations per fold and *something* will look good by
chance. This is the probability the Sharpe ratio survives that multiple-testing correction. Below
~0.5 the result is indistinguishable from noise dressed up by the search.

In [ ]:
dsr = deflated_sharpe(stats['sharpe'], n_trials=wf.n_evaluations, n_obs=int(stats['bars']), periods_per_year=PPY)
print(f"observed out-of-sample Sharpe : {stats['sharpe']:.2f}")
print(f"configurations evaluated      : {wf.n_evaluations:,}")
print(f"deflated Sharpe probability   : {dsr:.2f}")

## 6. The strongest test: assets the method was never tuned on

Everything so far has been BTC, and BTC is the asset the pipeline was developed against. The way
to find out whether the *method* generalises - rather than whether these parameters fit this
chart - is to point it, completely unchanged, at other markets.

If the agent only works on BTC, it is a BTC artefact. If it works across several independent
markets, there is something real underneath.

In [ ]:
CROSS = ['ETH-USD', 'SOL-USD', 'LINK-USD', 'DOGE-USD']
cross_rows = []
for sym in CROSS:
    try:
        d = load_prices(sym, interval=INTERVAL, start=START_DATE, source='coinbase')
        if len(d) < wf_cfg.train_bars + wf_cfg.test_bars + 60:
            print(f'{sym}: only {len(d)} bars, skipping'); continue
        r = walk_forward(d, exec_cfg, replace(wf_cfg, verbose=False))
        s = r.stats(benchmark=buy_and_hold_equity(d, START_CAPITAL).reindex(r.equity.index))
        cross_rows.append({
            'symbol': sym, 'bars': int(s['bars']), 'final equity': s['final_equity'],
            'buy & hold': s['benchmark_final_equity'], 'sharpe': s['sharpe'],
            'max drawdown': s['max_drawdown'], 'reached target': bool(s.get('target_hit', 0)),
        })
        print(f"{sym}: ${s['final_equity']:,.0f}  (buy & hold ${s['benchmark_final_equity']:,.0f})  sharpe {s['sharpe']:.2f}")
    except Exception as exc:
        print(f'{sym}: {type(exc).__name__}: {exc}')

cross = pd.DataFrame(cross_rows)
display(cross.style.format({'final equity': '${:,.0f}', 'buy & hold': '${:,.0f}',
                            'sharpe': '{:.2f}', 'max drawdown': '{:.1%}'}))

## 7. Spreading the $100 across several markets

One asset means one story. A portfolio shares the same $100 across several, sized by inverse
volatility so the wildest coin does not quietly become the whole book. Diversification will not
raise the headline return, but it usually raises return *per unit of drawdown* - and on a path to
a 10x, drawdown is what kills you before you arrive.

In [ ]:
universe = load_universe(['BTC-USD', 'ETH-USD', 'LINK-USD'], interval=INTERVAL,
                         start=START_DATE, source='coinbase')
panel = align_universe(universe)
print(f"{len(panel)} symbols aligned on {len(panel['BTC-USD']):,} common bars "
      f"({panel['BTC-USD'].index[0].date()} -> {panel['BTC-USD'].index[-1].date()})")

wf_pf = walk_forward(panel, exec_cfg, replace(wf_cfg, verbose=True))
pf_stats = wf_pf.stats()
print()
print(format_summary(pf_stats, 'PORTFOLIO walk-forward (out of sample)'))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
plot_equity(wf.equity, target=TARGET, initial=START_CAPITAL, title='Single asset vs portfolio', ax=ax)
ax.plot(wf_pf.equity.index, wf_pf.equity.values, color='#1baf7a', linewidth=2.0, label='Portfolio')
ax.legend(frameon=False, loc='upper left', fontsize=9)
plt.show()

compare = pd.DataFrame([
    {'run': f'{SYMBOL} only', **{k: stats[k] for k in ['final_equity','sharpe','max_drawdown','calmar']}},
    {'run': 'portfolio', **{k: pf_stats[k] for k in ['final_equity','sharpe','max_drawdown','calmar']}},
])
display(compare.style.format({'final_equity': '${:,.0f}', 'sharpe': '{:.2f}',
                              'max_drawdown': '{:.1%}', 'calmar': '{:.2f}'}))

## 8. How hard can you push toward $1,000?

The obvious way to reach the target faster is more leverage. The obvious way to reach $0 faster
is also more leverage. This sweep makes the trade explicit: for each risk setting it reports what
the account ended at, how deep the hole got, and - from the bootstrap - how often that same edge
ends in ruin.

Read the `p(ruin)` column before the `final equity` column. A path that reaches $1,000 in half the
cases and zero in the other half is not twice as good as a slower one; for a single account that
you only get to run once, it is worse.

In [ ]:
push_rows = []
for target_vol, max_lev in [(0.30, 1.0), (0.50, 1.5), (0.50, 2.0), (0.80, 2.0), (0.80, 3.0), (1.20, 3.0)]:
    ec = ExecutionConfig(initial_capital=START_CAPITAL, target_equity=TARGET, fee_bps=10.0,
                         slippage_bps=5.0, max_leverage=max_lev, periods_per_year=PPY)
    space = dict(DEFAULT_SEARCH_SPACE)
    space['target_vol'] = [target_vol]
    space['max_leverage'] = [max_lev]
    r = walk_forward(prices, ec, replace(wf_cfg, n_candidates=60, verbose=False), space)
    s = r.stats()
    m = monte_carlo_paths(r.returns, initial_capital=START_CAPITAL, target=TARGET, n_paths=2000, seed=3)
    push_rows.append({
        'target vol': target_vol, 'max leverage': max_lev,
        'final equity': s['final_equity'], 'max drawdown': s['max_drawdown'],
        'sharpe': s['sharpe'], 'reached target': bool(s.get('target_hit', 0)),
        'p(reach target)': m.get('p_hit_target', np.nan), 'p(ruin)': m.get('p_ruin', np.nan),
    })
    print(f"vol {target_vol:.2f} lev {max_lev:.1f}x -> ${s['final_equity']:,.0f}  "
          f"maxDD {s['max_drawdown']:.1%}  p(ruin) {m.get('p_ruin', float('nan')):.1%}")

push = pd.DataFrame(push_rows)
display(push.style.format({'final equity': '${:,.0f}', 'max drawdown': '{:.1%}', 'sharpe': '{:.2f}',
                           'p(reach target)': '{:.1%}', 'p(ruin)': '{:.1%}'}))

## 9. "Keep backtesting until it makes $1,000"

That instruction is a search, so `search_until_target` runs it as one - across markets and random
seeds - and keeps the receipt. It stops at the first attempt whose **out-of-sample** curve reaches
the target and reports how many attempts that took.

The attempt count is part of the result, not a footnote. If it takes twenty tries to find a market
and seed that reach $1,000, what has been discovered is mostly that twenty tries were run. One or
two, across markets that were never tuned on, is a different and much more interesting claim.

In [ ]:
from tradingagent.optimize import search_until_target

datasets = {'BTC-USD': prices}
for sym in CROSS:
    try:
        d = load_prices(sym, interval=INTERVAL, start=START_DATE, source='coinbase')
        if len(d) >= wf_cfg.train_bars + wf_cfg.test_bars + 60:
            datasets[sym] = d
    except Exception as exc:
        print(f'{sym}: {exc}')
datasets['portfolio'] = panel

search = search_until_target(
    datasets,
    exec_cfg,
    replace(wf_cfg, n_candidates=100),
    seeds=(1, 2, 3),
    stop_when_reached=True,
)
print()
print(search.summary())
display(search.attempts)

## 10. What would the agent do today?

The point of all of this. Given every bar up to the most recent close, this is the position the
agent wants to hold right now, what each model in the ensemble is saying, and how much weight the
blend is currently giving each one.

It prints an instruction. It does not place an order.

In [ ]:
fresh = load_prices(SYMBOL, interval=INTERVAL, start=START_DATE, source='coinbase', refresh=True)
rec = recommend(
    fresh,
    symbol=SYMBOL,
    equity=float(wf.equity.iloc[-1]),   # size against what the account actually has
    current_units=0.0,                  # set to what you currently hold
    agent_config=AgentConfig(periods_per_year=PPY),
    risk_config=risk_cfg,
)
print(rec)

## 11. What this does and does not show

Fill these in from your own run - the numbers above change every time the data updates.

**What the backtest supports**

- The walk-forward curve is out of sample: at every point, the settings being traded were chosen
  only from earlier data.
- The $1,000 target is reported as a *date on a path*, with the drawdown that had to be survived
  to get there - not as an average annual return.
- Costs, financing and slippage are charged on every fill, which is where most small-account
  strategies quietly die.

**What it does not support**

- **Survivorship of the asset itself.** BTC going up roughly 100x over the sample is the single
  biggest reason any long-biased agent looks good here. Compare against buy & hold every time,
  and note that buy & hold often wins on return while losing badly on drawdown.
- **My own iteration.** I developed this pipeline while looking at BTC. The cross-asset section is
  the corrective: judge the method by how it holds up on markets it was never tuned on.
- **Execution reality.** Fills are assumed at the open with fixed slippage. Real fills on thin alt
  pairs are worse, and a gap through a stop is worse again.
- **Regime change.** Every strategy here is a bet that some pattern that worked keeps working.
  The adaptive blend reduces how long a broken model stays funded; it cannot prevent a regime
  in which nothing in the panel works.

**If you intend to trade this with real money**

1. Paper-trade it first, for at least one full fold (six months), and compare the live equity to
   what the backtest said would happen. A divergence early is information, not bad luck.
2. Use the `p(ruin)` column, not the `final equity` column, to pick your risk setting.
3. Size so that the worst drawdown in the table above is one you would actually sit through. The
   most common way to lose with a working system is to turn it off at the bottom.
4. $100 is small enough that fees dominate. Fewer, larger positions beat frequent rebalancing -
   which is why `min_trade_frac` exists and defaults to 10% of equity.